***

Preparing Workspace

***

In [ ]:


## Importing packages ---

import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter
import time
import functools as ft
# pd.options.display.float_format = '{:.0f}'.format


## Setting file paths ---

user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'Census Data')
path_main = os.path.join(path_sp, 'Data')

if user == 'jfontes':
    path_git     = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
    path_config0 = os.path.join(path_git, 'config')
    path_code    = os.path.join(path_git, 'Data', 'Census')
    path_config  = os.path.join(path_code, 'config')


## User defined functions ---

exec(open(os.path.join(path_config0, 'Functions.py')).read())


## API key ---

# Obtain API Key from the following source 
# https://api.census.gov/data/key_signup.html
# Copy retrieved API key to .txt file for safe keeping
file_api = open(os.path.join(path_config, 'api_key.txt'))
api_key = file_api.read()
file_api.close()



***

Importing

***

In [ ]:


# Use URL to county fips mapping table
# Import county FIPS codes by state
url_puma_2020 = "https://www2.census.gov/geo/docs/maps-data/data/rel2020/2020_Census_Tract_to_2020_PUMA.txt"
url_puma_2010 = "https://www2.census.gov/geo/docs/maps-data/data/rel/2010_Census_Tract_to_2010_PUMA.txt"

df_puma_2020 = pd.read_csv(url_puma_2020, header = 0, sep = ',')
df_puma_2010 = pd.read_csv(url_puma_2010, header = 0, sep = ',')

df_puma_2020['Years'] = '2022-2031'
df_puma_2010['Years'] = '2012-2021'


df_puma = pd.concat([df_puma_2020, df_puma_2010])

df_puma.head()


# reformat FIPS fields
df_puma['PUMA5CE' ] = df_puma['PUMA5CE' ].astype(str).apply('{:0>5}'.format)
df_puma['TRACTCE' ] = df_puma['TRACTCE' ].astype(str).apply('{:0>6}'.format)
df_puma['COUNTYFP'] = df_puma['COUNTYFP'].astype(str).apply('{:0>3}'.format)
df_puma['STATEFP' ] = df_puma['STATEFP' ].astype(str).apply('{:0>2}'.format)

# show
df_puma.head()



In [ ]:


# Import PUMA Names
path_sdl = r'I:/Projects/Josh/Regional Monitoring'
df_puma_names_2020 = pd.read_excel(os.path.join(path_sdl, '2020_PUMA_Names.xlsx'))
df_puma_names_2010 = pd.read_excel(os.path.join(path_sdl, '2010_PUMA_Names.xlsx'))

df_puma_names_2020['Years'] = '2022-2031'
df_puma_names_2010['Years'] = '2012-2021'

df_puma_names = pd.concat([df_puma_names_2020, df_puma_names_2010])

df_puma_names['PUMA5CE'] = df_puma_names['PUMA5CE'].astype(str).apply('{:0>5}'.format)
df_puma_names['STATEFP'] = df_puma_names['STATEFP'].astype(str).apply('{:0>2}'.format)
df_puma_names.head()



In [ ]:


df_puma = df_puma.merge(df_puma_names, on = ['STATEFP', 'PUMA5CE', 'Years'], how = 'left')
df_puma.head()



***

Exporting

***

In [ ]:


# with pd.ExcelWriter(os.path.join(path_config0, 'Area Codes.xlsx'),mode='a',engine='openpyxl',if_sheet_exists='replace') as writer:
#             df_puma.to_excel(writer, index = False, sheet_name = 'PUMAcodes')

